# 05 — CTU-UHB External Validation — Feasibility Test

Initial feasibility test for CTU-UHB external validation. Extracts approximate 
ASTV/AC analogues from raw WFDB signal data to assess whether SisPorto-style 
features can be derived from raw FHR recordings. 

See Preliminary Report Section 4 for discussion of findings.

**Data**: not included in this repo — see `data/external/README.md` for download instructions.

In [9]:
import os
import wfdb
import numpy as np


# Update this path to wherever you've downloaded CTU-UHB records locally
# See data/external/README.md for download instructions
folder = '../data/external/ctu-chb-intrapartum-cardiotocography-database-1.0.0'

# list available records
hea_files = [f for f in os.listdir(folder) if f.endswith('.hea')]
print("Found records:", sorted(hea_files)[:10])

# load the first one to inspect structure
record_name = sorted(hea_files)[0].replace('.hea', '')
record_path = os.path.join(folder, record_name)
record = wfdb.rdrecord(record_path)

print("Signal names:", record.sig_name)
print("Sampling frequency:", record.fs)
print("Signal shape:", record.p_signal.shape)
print("First 10 values of channel 0 (FHR):", record.p_signal[:10, 0])

Found records: ['1001.hea', '1002.hea', '1003.hea', '1004.hea', '1005.hea', '1006.hea', '1007.hea', '1008.hea', '1009.hea', '1010.hea']
Signal names: ['FHR', 'UC']
Sampling frequency: 4
Signal shape: (19200, 2)
First 10 values of channel 0 (FHR): [150.5  150.5  151.   151.25 151.25 150.25 150.25 150.25 148.75 148.75]


In [10]:
def compute_astv_analogue(fhr, fs, window_sec=60):
    """Approximate Short-Term Variability: mean absolute difference 
    between consecutive samples, windowed."""
    window_size = int(window_sec * fs)
    fhr_clean = fhr[fhr > 50]  # remove dropout/zero artifacts
    diffs = np.abs(np.diff(fhr_clean))
    stv_windows = [diffs[i:i+window_size].mean() 
                   for i in range(0, len(diffs), window_size) if len(diffs[i:i+window_size]) > 0]
    return np.mean(stv_windows) if stv_windows else None


def compute_ac_analogue(fhr, fs, baseline_window=600):
    """Approximate Accelerations: count of rises >=15bpm above 
    local baseline lasting >=15 seconds."""
    baseline = np.median(fhr[fhr > 50])
    threshold = baseline + 15
    above = fhr > threshold
    min_samples = int(15 * fs)
    count, run_len = 0, 0
    for val in above:
        if val:
            run_len += 1
        else:
            if run_len >= min_samples:
                count += 1
            run_len = 0
    return count

In [11]:
# Run feasibility test on a sample of 3 records
record_ids = ['1001', '1002', '1003']

for rid in record_ids:
    record_path = os.path.join(folder, rid)
    rec = wfdb.rdrecord(record_path)
    fhr = rec.p_signal[:, 0]  # FHR channel
    fs = rec.fs
    
    astv = compute_astv_analogue(fhr, fs)
    ac = compute_ac_analogue(fhr, fs)
    
    print(f"Record {rid}: ASTV-analogue={astv:.2f}, AC-analogue={ac}")

Record 1001: ASTV-analogue=0.65, AC-analogue=10
Record 1002: ASTV-analogue=0.69, AC-analogue=8
Record 1003: ASTV-analogue=0.64, AC-analogue=5


## Finding

Signal-derived analogues can be computed directly from raw FHR data, but values 
are not on the same scale as UCI's SisPorto-derived ASTV (which reports variability 
as a percentage-of-time metric, not mean absolute bpm difference). Reconciling 
this requires replicating SisPorto's specific calculation methodology — identified 
as the primary remaining task for the Interim Report phase.

## Revisiting the ASTV/AC Mismatch

The initial feasibility test showed ASTV-analogue values (0.64–0.69) sitting far 
outside the UCI dataset's ASTV range (12–87), indicating the original function 
was structurally closer to UCI's *MSTV* feature than ASTV. 

SisPorto's ASTV is a percentage-of-time metric, not a mean. The revised 
functions below implement point-level classification instead of a windowed 
mean, using an abnormality threshold reported in CTG signal-processing 
literature (points classified as abnormal STV when beat-to-beat difference 
< 1bpm; abnormal LTV when 60-second window range ≤ 5bpm).

**Note on sourcing:** the primary SisPorto methodology paper (Ayres-de-Campos 
et al., 2000) is being requested via library/author channels to verify this 
threshold directly. This implementation is an interim, literature-informed 
replication pending that verification, and is documented as such.

In [12]:
def compute_astv_v2(fhr, fs, threshold_bpm=1.0):
    fhr_clean = fhr[fhr > 50]
    diffs = np.abs(np.diff(fhr_clean))
    abnormal_points = diffs < threshold_bpm
    return 100 * abnormal_points.mean()

def compute_altv_v2(fhr, fs, window_sec=60, range_threshold_bpm=5.0):
    fhr_clean = fhr[fhr > 50]
    window_size = int(window_sec * fs)
    abnormal_windows, total_windows = 0, 0
    for i in range(0, len(fhr_clean) - window_size, window_size):
        seg = fhr_clean[i:i+window_size]
        if len(seg) == 0:
            continue
        rng = seg.max() - seg.min()
        abnormal_windows += (rng <= range_threshold_bpm)
        total_windows += 1
    return 100 * abnormal_windows / total_windows if total_windows else None

In [13]:
for rid in record_ids:
    record_path = os.path.join(folder, rid)
    rec = wfdb.rdrecord(record_path)
    fhr = rec.p_signal[:, 0]
    fs = rec.fs

    astv_v2 = compute_astv_v2(fhr, fs)
    altv_v2 = compute_altv_v2(fhr, fs)

    print(f"Record {rid}: ASTV_v2={astv_v2:.2f}, ALTV_v2={altv_v2:.2f}")

Record 1001: ASTV_v2=80.86, ALTV_v2=0.00
Record 1002: ASTV_v2=81.10, ALTV_v2=1.52
Record 1003: ASTV_v2=72.64, ALTV_v2=0.00


In [14]:
import pandas as pd

# Run across a larger sample to check for genuine distribution patterns
sample_ids = sorted(hea_files)[:20]  # first 20 records
results = []

for f in sample_ids:
    rid = f.replace('.hea', '')
    record_path = os.path.join(folder, rid)
    try:
        rec = wfdb.rdrecord(record_path)
        fhr = rec.p_signal[:, 0]
        fs = rec.fs

        astv_v2 = compute_astv_v2(fhr, fs)
        altv_v2 = compute_altv_v2(fhr, fs)
        results.append({'record': rid, 'ASTV_v2': astv_v2, 'ALTV_v2': altv_v2})
    except Exception as e:
        print(f"Skipped {rid}: {e}")

results_df = pd.DataFrame(results)
print(results_df)
print("\n--- Summary stats ---")
print(results_df[['ASTV_v2', 'ALTV_v2']].describe())
print("\n--- For comparison, UCI's actual ASTV/ALTV ---")
print("UCI ASTV: mean=46.98, min=12, max=87")
print("UCI ALTV: mean=9.76, min=0, max=91")

   record    ASTV_v2   ALTV_v2
0    1001  80.855193  0.000000
1    1002  81.095495  1.515152
2    1003  72.640257  0.000000
3    1004  83.588753  0.000000
4    1005  78.419778  0.000000
5    1006  75.255704  1.923077
6    1007  75.527634  4.000000
7    1008  82.468297  3.174603
8    1009  82.988386  0.000000
9    1010  79.310345  3.225806
10   1011  69.113254  3.278689
11   1012  81.761063  2.173913
12   1013  76.709837  0.000000
13   1014  82.993701  1.492537
14   1015  81.504747  1.724138
15   1016  77.185583  3.703704
16   1017  78.139301  0.000000
17   1018  70.657357  0.000000
18   1019  76.078262  0.000000
19   1020  70.711877  2.898551

--- Summary stats ---
         ASTV_v2    ALTV_v2
count  20.000000  20.000000
mean   77.850241   1.455508
std     4.478280   1.502678
min    69.113254   0.000000
25%    75.459652   0.000000
50%    78.279539   1.503844
75%    81.568826   2.967564
max    83.588753   4.000000

--- For comparison, UCI's actual ASTV/ALTV ---
UCI ASTV: mean=46.98, min=

In [15]:
for thresh in [0.3, 0.5, 0.75, 1.0, 1.5, 2.0]:
    vals = []
    for f in sample_ids:
        rid = f.replace('.hea', '')
        rec = wfdb.rdrecord(os.path.join(folder, rid))
        fhr = rec.p_signal[:, 0]
        vals.append(compute_astv_v2(fhr, rec.fs, threshold_bpm=thresh))
    vals = pd.Series(vals)
    print(f"threshold={thresh}: mean={vals.mean():.1f}, min={vals.min():.1f}, max={vals.max():.1f}, std={vals.std():.1f}")

threshold=0.3: mean=62.6, min=53.4, max=70.0, std=4.8
threshold=0.5: mean=62.6, min=53.4, max=70.0, std=4.8
threshold=0.75: mean=69.6, min=54.0, max=77.2, std=6.2
threshold=1.0: mean=77.9, min=69.1, max=83.6, std=4.5
threshold=1.5: mean=85.5, min=75.9, max=90.9, std=4.6
threshold=2.0: mean=90.3, min=81.5, max=94.4, std=3.7


In [16]:
def compute_astv_v3(fhr, fs, threshold_bpm=1.0, lag_sec=1.0):
    fhr_clean = fhr[fhr > 50]
    lag = int(lag_sec * fs)
    diffs = np.abs(fhr_clean[lag:] - fhr_clean[:-lag])
    abnormal_points = diffs < threshold_bpm
    return 100 * abnormal_points.mean()

for lag in [0.25, 0.5, 1.0, 2.0]:
    vals = []
    for f in sample_ids:
        rid = f.replace('.hea', '')
        rec = wfdb.rdrecord(os.path.join(folder, rid))
        fhr = rec.p_signal[:, 0]
        vals.append(compute_astv_v3(fhr, rec.fs, threshold_bpm=1.0, lag_sec=lag))
    vals = pd.Series(vals)
    print(f"lag={lag}s: mean={vals.mean():.1f}, min={vals.min():.1f}, max={vals.max():.1f}, std={vals.std():.1f}")

lag=0.25s: mean=77.9, min=69.1, max=83.6, std=4.5
lag=0.5s: mean=59.2, min=41.1, max=69.6, std=7.5
lag=1.0s: mean=44.6, min=25.8, max=59.2, std=7.2
lag=2.0s: mean=31.0, min=17.6, max=44.2, std=5.8


In [17]:
def compute_astv_final(fhr, fs, threshold_bpm=1.0, lag_sec=1.0):
    """
    ASTV analogue: percentage of 1-second-apart FHR sample pairs classified as
    'abnormal' short-term variability (difference < 1bpm).
    
    Selected after threshold sensitivity testing (0.3-2.0bpm) and lag testing
    (0.25-2.0s) against a 20-record CTU-UHB sample. This configuration produced
    a mean (44.6) closest to UCI's ASTV mean (46.98), compared to adjacent-sample
    comparison which was compressed and skewed high (mean ~78 regardless of 
    threshold). Spread remains narrower than UCI's (std=7.2 vs 17.2), a 
    documented limitation given SisPorto's undisclosed proprietary methodology.
    """
    fhr_clean = fhr[fhr > 50]
    lag = int(lag_sec * fs)
    diffs = np.abs(fhr_clean[lag:] - fhr_clean[:-lag])
    abnormal_points = diffs < threshold_bpm
    return 100 * abnormal_points.mean()

## Threshold and Lag Sensitivity Testing — Final Configuration

Initial adjacent-sample comparisons (lag=0.25s, matching the raw 4Hz sampling 
rate) produced ASTV values compressed near the top of UCI's scale (mean ~78, 
std ~4.5) regardless of the classification threshold tested (0.3–2.0bpm), 
suggesting the raw sampling interval was too fine-grained to capture genuine 
inter-record variability differences.

Testing larger sample lags found that comparing FHR values 1 second apart 
produced a mean (44.6) closely matching UCI's ASTV mean (46.98), a substantial 
improvement over the adjacent-sample approach. This configuration 
(threshold=1bpm, lag=1s) was selected as the final ASTV-analogue methodology. 
The spread remains narrower than UCI's (std=7.2 vs 17.2), which is documented 
as a limitation of this literature-informed replication, given SisPorto's exact 
proprietary computation is not fully published.

## Citation Update — Corroborating Source Found (Interim Report phase)

The ASTV threshold used above (points classified as abnormal STV when the
difference between FHR samples is < 1bpm) is now corroborated by a stronger,
directly-relevant source than general "CTG signal-processing literature":

**Costa M, Xavier M, Nunes I, Henriques TS. "Fetal Heart Rate Fragmentation."
Front Pediatr. 2021;9:662101.**

This paper states SisPorto's rule explicitly, by name: *"Abnormal short-term
variability (STV) — percentage of subsequent FHR signals differing less than
1 bpm"* and *"Abnormal long-term variability (LTV) — percentage of FHR
signals with a difference between the minimum and maximum values in a 1 min
window lower than 5 bpm."* Both match the thresholds used in `compute_astv_final`
and `compute_altv_v2` exactly.

Critically, this paper also ran the real SisPorto software on **this same
CTU-UHB database** and reports its actual output: median abnormal STV
44-46%, median abnormal LTV ~2%. The `lag=1s` configuration selected above
(mean=44.6, chosen for matching UCI's mean=46.98) is now independently
corroborated by a second, more relevant benchmark — real SisPorto output on
CTU-UHB itself, not just UCI's differently-sourced distribution. Two
independent checks landing in the same range strengthens confidence that
`lag=1s, threshold=1bpm` is a reasonable operational match to SisPorto's
behaviour, even without access to the primary methodology paper
(Ayres-de-Campos et al. 2000, still unavailable via library/author request
at time of writing).

This does not change the implementation above — it upgrades the citation
supporting it from a general literature reference to a specific,
SisPorto-attributed, same-dataset-validated source.